# Task 10: Denoising Diffusion Probabilistic Model (DDPM) Forward & Reverse Latent Optimization

## Forward Formula
$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon$$


In [1]:
import torch

# DDPM module computing linear variance schedule and forward noising trajectory
class DDPM:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.timesteps = timesteps
        self.betas = torch.linspace(beta_start, beta_end, timesteps)
        self.alphas_cumprod = torch.cumprod(1.0 - self.betas, dim=0)

    def q_sample(self, x_0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_0)
        sqrt_alpha_bar = torch.sqrt(self.alphas_cumprod[t]).view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - self.alphas_cumprod[t]).view(-1, 1, 1, 1)
        return sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * noise


In [2]:
# Simulate forward diffusion process across timesteps t in [0, 250, 500, 1000]
ddpm = DDPM(timesteps=1000)
clean_image = torch.randn(1, 1, 28, 28)

timesteps_to_test = [0, 250, 500, 999]

print(f"{'Timestep t':<12} | {'Alpha_bar':<12} | {'Noised Tensor Mean':<20} | {'Noised Tensor Std':<20}")
print("-" * 70)

for t in timesteps_to_test:
    t_tensor = torch.tensor([t])
    noised = ddpm.q_sample(clean_image, t_tensor)
    alpha_bar = ddpm.alphas_cumprod[t].item()
    print(f"{t:<12} | {alpha_bar:<12.4f} | {noised.mean().item():<20.4f} | {noised.std().item():<20.4f}")


Timestep t   | Alpha_bar    | Noised Tensor Mean   | Noised Tensor Std   
----------------------------------------------------------------------
0            | 0.9999       | -0.0177              | 0.9520              
250          | 0.5214       | -0.0202              | 0.9605              
500          | 0.0778       | 0.0239               | 0.9640              
999          | 0.0000       | -0.0862              | 1.0100              
